# Waymo Open Motion Dataset: Read and Inspect Locally

This notebook is adapted from the official Waymo motion tutorial and tailored to your local dataset path:

- Dataset root: `/mnt/disk/data/public/waymo`
- Goal: learn how to open TFRecord files, parse `Scenario` protos, and inspect tracks/map data

Reference tutorial: https://github.com/waymo-research/waymo-open-dataset/blob/master/tutorial/tutorial_motion.ipynb

## 1) Installation (run once per environment)

If your environment already has Waymo Open Dataset + TensorFlow, you can skip this cell.

In [ ]:
# Uncomment if needed:
# uv sync

## 2) Imports and global config

In [1]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from waymo_open_dataset.protos import scenario_pb2

DATASET_ROOT = Path('/mnt/disk/data/public/waymo')
print('TensorFlow:', tf.__version__)
print('Dataset root exists:', DATASET_ROOT.exists())
print('Dataset root:', DATASET_ROOT)

2026-04-09 13:47:40.955158: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 13:47:40.957132: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-09 13:47:40.985042: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-09 13:47:40.986191: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-09 13:47:44.200965: W tensorflow/compiler/tf2t

TensorFlow: 2.12.0
Dataset root exists: False
Dataset root: /mnt/disk/data/public/waymo


## 3) Discover motion TFRecord files

This searches recursively under `/mnt/disk/data/public/waymo` for `.tfrecord` files.

In [ ]:
all_tfrecords = sorted(DATASET_ROOT.rglob('*.tfrecord*'))
print('Found', len(all_tfrecords), 'TFRecord-like files')
for p in all_tfrecords[:20]:
    print('-', p)
if len(all_tfrecords) > 20:
    print('... (showing first 20)')

## 4) Pick one file and parse a scenario

Each record in a motion TFRecord is serialized as `waymo_open_dataset.protos.Scenario`.

In [ ]:
if not all_tfrecords:
    raise FileNotFoundError(f'No TFRecord files found under {DATASET_ROOT}')

TFRECORD_PATH = all_tfrecords[0]  # change to any file you want
print('Using TFRecord:', TFRECORD_PATH)

def parse_scenario(serialized_bytes):
    scenario = scenario_pb2.Scenario()
    scenario.ParseFromString(serialized_bytes)
    return scenario

dataset = tf.data.TFRecordDataset([str(TFRECORD_PATH)], compression_type='')
first_record = next(iter(dataset.as_numpy_iterator()))
scenario = parse_scenario(first_record)
print('Parsed scenario_id:', scenario.scenario_id)

In [ ]:
print('num_timestamps:', scenario.timestamps_seconds.__len__())
print('num_tracks:', len(scenario.tracks))
print('num_map_features:', len(scenario.map_features))
print('num_dynamic_map_states:', len(scenario.dynamic_map_states))
print('sdc_track_index:', scenario.sdc_track_index)

## 5) Convert track states to a table

This flattens every `(track, timestep)` into one row for quick inspection in pandas.

In [ ]:
def tracks_to_dataframe(scenario_msg):
    rows = []
    for t_idx, trk in enumerate(scenario_msg.tracks):
        for s_idx, st in enumerate(trk.states):
            rows.append({
                'track_array_index': t_idx,
                'track_id': trk.id,
                'object_type': trk.object_type,
                'step': s_idx,
                'valid': bool(st.valid),
                'x': st.center_x,
                'y': st.center_y,
                'z': st.center_z,
                'length': st.length,
                'width': st.width,
                'height': st.height,
                'heading': st.heading,
                'velocity_x': st.velocity_x,
                'velocity_y': st.velocity_y,
            })
    return pd.DataFrame(rows)

tracks_df = tracks_to_dataframe(scenario)
print('rows:', len(tracks_df))
tracks_df.head(10)

In [ ]:
valid_df = tracks_df[tracks_df['valid']]
print('Unique tracks with at least one valid state:', valid_df['track_id'].nunique())
print('Counts by object_type:')
print(valid_df.groupby('object_type')['track_id'].nunique().sort_values(ascending=False))

## 6) Inspect map feature types

In [ ]:
def map_feature_kind(feature):
    # Returns which oneof field is set for this map feature.
    return feature.WhichOneof('feature_data')

map_kinds = pd.Series([map_feature_kind(f) for f in scenario.map_features])
map_kinds.value_counts(dropna=False)

## 7) Quick 2D trajectory plot

This plots valid XY history for a random subset of tracks in one scenario.

In [ ]:
def plot_random_tracks(df, num_tracks=20, seed=0):
    rng = random.Random(seed)
    valid = df[df['valid']].copy()
    track_ids = valid['track_id'].drop_duplicates().tolist()
    if not track_ids:
        raise ValueError('No valid tracks to plot.')
    chosen = set(rng.sample(track_ids, k=min(num_tracks, len(track_ids))))
    subset = valid[valid['track_id'].isin(chosen)].sort_values(['track_id', 'step'])

    fig, ax = plt.subplots(figsize=(8, 8))
    for tid, g in subset.groupby('track_id'):
        ax.plot(g['x'], g['y'], linewidth=1.2, alpha=0.8)
        ax.scatter(g['x'].iloc[-1], g['y'].iloc[-1], s=16)

    ax.set_title(f'Random valid trajectories (n={len(chosen)})')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    plt.show()

plot_random_tracks(tracks_df, num_tracks=20, seed=42)

## 8) Iterate over multiple scenarios

Use this pattern when building training/evaluation pipelines.

In [ ]:
def iter_scenarios(tfrecord_path, max_scenarios=None):
    ds = tf.data.TFRecordDataset([str(tfrecord_path)], compression_type='')
    for i, raw in enumerate(ds):
        if max_scenarios is not None and i >= max_scenarios:
            break
        s = scenario_pb2.Scenario()
        s.ParseFromString(raw.numpy())
        yield s

for i, s in enumerate(iter_scenarios(TFRECORD_PATH, max_scenarios=3), start=1):
    print(f'{i}. scenario_id={s.scenario_id}, tracks={len(s.tracks)}, map_features={len(s.map_features)}')

## 9) Next steps

- Change `TFRECORD_PATH` to target specific train/val/test shards.
- Extend extraction with lane centerlines / traffic lights for model features.
- Wrap `iter_scenarios` into a PyTorch or TensorFlow dataloader pipeline.